> 在线讲义：[第 19 章 · 从矩阵乘法认识加速器：算多少、搬多少、耗多少电](https://wikieden.github.io/ai-infra-from-constraints/lessons/19-accelerator-basics.html) · 本书《从约束推导 AI Infra》

# 第 19 章 · 从矩阵乘法认识加速器：算多少、搬多少、耗多少电

配套讲义：仓库里 `ai-infra/notes/lessons/19-accelerator-basics.md`（附录最后一节"代码对照"）

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的 Python 写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。纯 Python，不需要 GPU，也不下载模型。练习题不在这里算，先在讲义里自己做。

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。

| 讲义 | 公式 | 写法 | 跑出来 |
|---|---|---|---|
| 第一步 Q 投影 | $F$、$V$、$I$ | `proj_cost(1)`、`proj_cost(256)` | 约 1、约 228 FLOPs/byte；上限 2048 |
| 第三步 Amdahl | 60 / 40 μs | `amdahl_us(60, 40, s)` | 70、55、40 μs |
| 第四步 电压与功率密度 | 平方、比值 | `(0.55/0.85)**2`、`0.75/0.60` | 0.42、1.25 |
| 第四步 decode 能量 | 字节 × pJ | `decode_energy()` | 0.481 + 0.038 + 0.015 = 0.534 J |
| 第五步 HBM | $1024r/8$ | `hbm_stack_gbs(5.24)` | 约 670 GB/s；5 堆约 3350 |
| 第五步 功率封顶 | $b$、$b/e$ | `power_budget()`、`sustained(1.5)` | 0.60 pJ/FLOP；0.67 / 0.87 |

Q 投影形状、60/40 μs 算例、LogicFolding 的电压与面积、能耗表、decode 能量账、HBM 与 TDP 取自 AII 第 4 章 §4.1。

## 1. 先把要用的东西定义好

这几格只是定义，运行后没有输出也正常。

In [ ]:
PJ_PER_BYTE = {"KB 级 SRAM": 1.25, "MB 级 SRAM": 12.5, "NVLink-C2C": 10.4, "HBM2": 31.76, "LPDDR": 160.0}
PJ_PER_FLOP = 0.75  # 16 位浮点乘加 1.5 pJ / 2 FLOPs

**函数 `proj_cost`**：矩阵乘法 [M,K]×[K,N]：(运算 FLOPs, 片外读写字节, 算术强度)。

In [ ]:
def proj_cost(M: int, K: int = 4096, N: int = 4096, b: int = 2) -> tuple[int, int, float]:
    """矩阵乘法 [M,K]×[K,N]：(运算 FLOPs, 片外读写字节, 算术强度)。"""
    F = 2 * M * K * N  # 运算
    V = b * (M * K + K * N + M * N)  # 输入 + 权重 + 输出，各读写一次
    return F, V, F / V

**函数 `intensity_limit`**：M → ∞ 时算术强度的上限 KN/(K+N)。

In [ ]:
def intensity_limit(K: int = 4096, N: int = 4096) -> float:
    """M → ∞ 时算术强度的上限 KN/(K+N)。"""
    return K * N / (K + N)

**函数 `amdahl_us`**：矩阵部分加速 s 倍后的总时间（μs）。

In [ ]:
def amdahl_us(matrix_us: float, other_us: float, s: float) -> float:
    """矩阵部分加速 s 倍后的总时间（μs）。"""
    return matrix_us / s + other_us

**函数 `decode_energy`**：一步 decode 的 (权重, KV, 计算) 能量（J），HBM2 31.76 pJ/byte，乘加 0.75 pJ/FLOP。

In [ ]:
def decode_energy(w_gb: float = 15.137, kv_gb: float = 1.208, gflops: float = 19.97) -> tuple[float, float, float]:
    """一步 decode 的 (权重, KV, 计算) 能量（J），HBM2 31.76 pJ/byte，乘加 0.75 pJ/FLOP。"""
    hbm = PJ_PER_BYTE["HBM2"] * 1e-12
    return w_gb * 1e9 * hbm, kv_gb * 1e9 * hbm, gflops * 1e9 * PJ_PER_FLOP * 1e-12

**函数 `energy_per_token`**：batch 为 B 时每 token 能量：权重读一次服务 B 个 token。

In [ ]:
def energy_per_token(B: int) -> float:
    """batch 为 B 时每 token 能量：权重读一次服务 B 个 token。"""
    w, kv, c = decode_energy()
    return w / B + kv + c

**函数 `hbm_stack_gbs`**：每堆 HBM 带宽（GB/s）= 1024 根引脚 × 速率 / 8。

In [ ]:
def hbm_stack_gbs(pin_gbit: float) -> float:
    """每堆 HBM 带宽（GB/s）= 1024 根引脚 × 速率 / 8。"""
    return 1024 * pin_gbit / 8

**函数 `power_budget`**：峰值速率下每 FLOP 的能量预算（pJ）=（TDP − HBM 功率）/ 峰值算力。

In [ ]:
def power_budget(tdp_w: float = 700, bw_tbs: float = 3.35, peak_tflops: float = 989.4) -> float:
    """峰值速率下每 FLOP 的能量预算（pJ）=（TDP − HBM 功率）/ 峰值算力。"""
    mem_w = bw_tbs * 1e12 * PJ_PER_BYTE["HBM2"] * 1e-12
    return (tdp_w - mem_w) / (peak_tflops * 1e12) * 1e12

**函数 `sustained`**：每 FLOP 能耗超出预算时的持续频率比：(电压不变 b/e, 电压同步下降 (b/e)^(1/3))。

In [ ]:
def sustained(e_over_b: float) -> tuple[float, float]:
    """每 FLOP 能耗超出预算时的持续频率比：(电压不变 b/e, 电压同步下降 (b/e)^(1/3))。"""
    r = min(1.0, 1 / e_over_b)
    return r, r ** (1 / 3)

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("\n第一步：Qwen3-8B 的 Q 投影")
for M in (1, 256):
    F, V, I = proj_cost(M)
    print(f"  M = {M:3d}：运算 {F / 1e9:.4g} GFLOPs，访问 {V / 2**20:.2f} MiB，算术强度 {I:.1f}")
print(f"  上限 KN/(K+N) = {intensity_limit():.0f}")

In [ ]:
print("\n第三步：60 μs 矩阵 + 40 μs 其余")
for s in (1, 2, 4, float("inf")):
    t = amdahl_us(60, 40, s)
    print(f"  矩阵加速 {s} 倍：{t:.0f} μs，加速比 {100 / t:.2f}")

In [ ]:
print("\n第四步：功耗与能量")
print(f"  电压 0.85 → 0.55 V：功耗 ×{(0.55 / 0.85) ** 2:.2f}；功率密度 0.75 / 0.60 = {0.75 / 0.60:.2f}")
w, kv, c = decode_energy()
print(f"  decode 一步：权重 {w:.3f} J + KV {kv:.3f} J + 计算 {c:.3f} J = {w + kv + c:.3f} J，权重占 {w / (w + kv + c):.0%}")
print(f"  batch 32 时每 token {energy_per_token(32):.3f} J")

In [ ]:
print("\n第五步：HBM 与功率封顶")
per = hbm_stack_gbs(5.24)
print(f"  H100：每堆 {per:.0f} GB/s，5 堆 {5 * per:,.0f} GB/s")
print(f"  每 FLOP 能量预算 {power_budget():.2f} pJ；e/b = 1.5 时持续频率 {sustained(1.5)[0]:.2f} / {sustained(1.5)[1]:.2f}")

## 3. 代码里才看得到的

1. `proj_cost` 假设权重只从片外读一次；如果分块太小，权重会被读多次，真实访问量比 $V$ 大（第 26 章）。
2. 能量账里 HBM 用的是 28 nm HBM2 模型、乘加用的是 45 nm 数据，两者工艺不同，所以只用来比量级（原文 L173）。
3. `sustained` 两种情况：电压不变时功率随频率线性降，所以是 $b/e$；电压同步降时功率是频率的三次方，才取立方根。

## 4. 自己改一改

- 用 `proj_cost` 扫 $M$ = 1、4、16、64、256、1024、4096，画出 $I(M)$。
- 把 `decode_energy` 的 HBM 换成 LPDDR（160 pJ/byte），看一步多少焦耳。
- 用 `hbm_stack_gbs` 算一算 HBM3E 9.6 Gbit/s 的每堆带宽。

在下面的空格子里写，或者直接改上面的格子。

In [ ]:
# 在这里写你的代码

## 口诀

> 标准答案一份，学生越多摊得越薄，作业本还得一本本收；
> 工厂里下单、仓库、料架、机床、质检、叉车，各管一段；
> 炒锅多了备菜跟不上，缓冲比计算还占地；
> 水管越长灌水越费，decode 的电九成花在搬权重；
> 地皮有上限，水管看周长，总电闸限开机。